# Model açıklamaları için hata ayıklama testleri

Bölüm 3 · Yorumlanabilirliğin amaçları · Adebayo ve ark. (2020)

## 1. Özgün IDX arşivleri ve CNN

İki kaynak için aynı piksel ölçeği ve aynı beş evrişimli, üç tam bağlı katmanlı CNN kullanılır.

In [1]:
import gzip
import hashlib
import platform
from pathlib import Path
from urllib.request import urlopen

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy.stats import rankdata
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(4)
ROOT = Path.cwd()
FIG = ROOT / "sekiller"
FIG.mkdir(exist_ok=True)
FILES = ("train-images-idx3-ubyte.gz", "train-labels-idx1-ubyte.gz",
         "t10k-images-idx3-ubyte.gz", "t10k-labels-idx1-ubyte.gz")
SOURCES = {
    "MNIST": ("https://storage.googleapis.com/cvdf-datasets/mnist/",
              ("f68b3c2dcbeaaa9fbdd348bbdeb94873", "d53e105ee54ea40749a09fcbcd1e9432",
               "9fb629c4189551a2d022fa330f9573f3", "ec29112dd5afa0611ce80d1b7f02629c")),
    "Fashion-MNIST": ("https://raw.githubusercontent.com/zalandoresearch/fashion-mnist/master/data/fashion/",
              ("8d4fb7e6c68d591d4c3dfef9ec88bf0d", "25c81989df183df01b3e8a0aad5dffbe",
               "bef4ecab320f06d8554ea6380940ec79", "bb300cfdad3c16e7a12a480ee83cd310")),
}


def load_dataset(name):
    base, hashes = SOURCES[name]
    arrays = []
    for index, (filename, expected) in enumerate(zip(FILES, hashes)):
        path = ROOT / "data" / "raw" / name / filename
        path.parent.mkdir(parents=True, exist_ok=True)
        raw = path.read_bytes() if path.is_file() else None
        if raw is None or hashlib.md5(raw).hexdigest() != expected:
            try:
                with urlopen(base + filename, timeout=90) as response:
                    raw = response.read()
            except OSError as exc:
                raise RuntimeError(f"Veri indirilemedi: {base + filename}") from exc
            if hashlib.md5(raw).hexdigest() != expected:
                raise ValueError(f"Arşiv sağlama toplamı uyuşmuyor: {filename}")
            path.write_bytes(raw)
        data = gzip.decompress(raw)
        n = 60000 if index < 2 else 10000
        images = index % 2 == 0
        header = (2051, n, 28, 28) if images else (2049, n)
        assert tuple(np.frombuffer(data, dtype=">i4", count=len(header))) == header
        pixels = np.frombuffer(data, dtype=np.uint8, offset=4 * len(header)).copy()
        tensor = torch.from_numpy(pixels)
        arrays.append(tensor.reshape(n, 1, 28, 28).float().div_(255) if images else tensor.long())
    return arrays


class GuidedReLU(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        return x.clamp_min(0)

    @staticmethod
    def backward(ctx, upstream):
        x, = ctx.saved_tensors
        return upstream.clamp_min(0) * (x > 0)


class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        widths = (1, 16, 16, 32, 32, 48)
        self.convs = nn.ModuleList(nn.Conv2d(a, b, 3, padding=1)
                                   for a, b in zip(widths[:-1], widths[1:]))
        self.hidden = nn.ModuleList([nn.Linear(7 * 7 * 48, 128), nn.Linear(128, 64)])
        self.out = nn.Linear(64, 10)
        self.to(memory_format=torch.channels_last)

    def forward(self, x, guided=False):
        relu = GuidedReLU.apply if guided else F.relu
        for i, conv in enumerate(self.convs):
            x = relu(conv(x))
            if i in (1, 4):
                x = F.max_pool2d(x, 2)
        x = x.flatten(1)
        for layer in self.hidden:
            x = relu(layer(x))
        return self.out(x)


@torch.no_grad()
def predictions(model, x):
    return torch.cat([model(batch) for batch in x.split(128)])


def train(x, y, epochs):
    torch.manual_seed(17)
    model = CNN()
    order = torch.Generator().manual_seed(23)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    history = []
    for epoch in range(1, epochs + 1):
        correct, total_loss = 0, 0.0
        for idx in torch.randperm(len(y), generator=order).split(128):
            optimizer.zero_grad()
            logits = model(x[idx])
            loss = F.cross_entropy(logits, y[idx])
            loss.backward()
            optimizer.step()
            correct += (logits.argmax(1) == y[idx]).sum().item()
            total_loss += loss.item() * len(idx)
        history.append((epoch, correct / len(y), total_loss / len(y)))
        if epoch % 10 == 0:
            print(f"Dönem {epoch}: {correct / len(y):.3f}", flush=True)
    return model.eval(), pd.DataFrame(history, columns=["Dönem", "Eğitim doğruluğu", "Kayıp"])


def gradient(model, x, target, guided=False):
    z = x.detach().requires_grad_(True)
    score = model(z, guided=guided).gather(1, target[:, None]).sum()
    return torch.autograd.grad(score, z)[0]


def integrated(model, x, target, steps=24):
    total = torch.zeros_like(x)
    for i in range(steps + 1):
        weight = 0.5 if i in (0, steps) else 1.0
        total += weight * gradient(model, x * (i / steps), target)
    return x * total / steps


def rank_similarity(a, b):
    ra, rb = rankdata(a), rankdata(b)
    if np.ptp(ra) == 0 or np.ptp(rb) == 0:
        return np.nan
    return np.corrcoef(ra, rb)[0, 1]


def save_figure(fig, name):
    fig.tight_layout()
    fig.savefig(FIG / name, dpi=150, bbox_inches="tight")
    plt.close(fig)


def environment():
    return f"Python {platform.python_version()}, PyTorch {torch.__version__}, CPU"


## 2. Model eğitimi ve alan içi performans

Her model yalnız kendi kaynağının eğitim görüntülerini görür. Test görüntüleri eğitimde kullanılmaz.

In [2]:
from IPython.display import Markdown, display
from scipy.ndimage import gaussian_filter

domains = ["Fashion-MNIST", "MNIST"]
datasets = {name: load_dataset(name) for name in domains}
models, histories, test_rows = {}, {}, []
for name in domains:
    train_x, train_y, test_x, test_y = datasets[name]
    chosen = np.random.default_rng(41).choice(len(train_y), 12000, replace=False)
    models[name], histories[name] = train(train_x[chosen], train_y[chosen], 8)
    accuracy = (predictions(models[name], test_x).argmax(1) == test_y).float().mean().item()
    test_rows.append((name, len(train_y), len(chosen), len(test_y), accuracy))
training = pd.DataFrame(test_rows, columns=["Modelin eğitim alanı", "Özgün eğitim",
    "Kullanılan eğitim", "Özgün test", "Alan içi test doğruluğu"])
display(training)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for name, history in histories.items():
    for ax, metric in zip(axes, ["Eğitim doğruluğu", "Kayıp"]):
        ax.plot(history["Dönem"], history[metric], label=name)
        ax.set(xlabel="Dönem", ylabel=metric)
        ax.legend()
save_figure(fig, "01-egitim.png")

,Modelin eğitim alanı,Özgün eğitim,Kullanılan eğitim,Özgün test,Alan içi test doğruluğu
0,Fashion-MNIST,60000,12000,10000,0.8539
1,MNIST,60000,12000,10000,0.9760


## 3. Aynı test görüntüsünde alan içi ve alan dışı atıflar

Her model kendi en yüksek logitini açıklar. Aynı görüntünün pikselleri iki koşulda sabittir. Etiket uzayları farklı olduğu için OOD girdide sınıflandırma doğruluğu tanımlanmaz.

In [3]:
methods = ["Gradyan", "SmoothGrad", "SmoothGrad-SQ", "VarGrad",
           "Girdi × gradyan", "IG", "Guided Backprop"]

def explain(model, x, target, seed):
    g = gradient(model, x, target).abs()
    generator = torch.Generator().manual_seed(seed)
    flat = x.flatten(1)
    sigma = (0.15 * (flat.amax(1) - flat.amin(1)))[:, None, None, None]
    noisy = torch.stack([
        gradient(model, x + sigma * torch.randn(x.shape, generator=generator), target)
        for _ in range(50)
    ])
    smooth = noisy.mean(0)
    maps = [g, smooth, smooth.square(), noisy.var(0, unbiased=False),
            x * g, integrated(model, x, target), gradient(model, x, target, guided=True)]
    return {name: array[:, 0].numpy() for name, array in zip(methods, maps)}


def normalize(a):
    return a / max(np.abs(a).max(), 1e-12)


SSIM_RADIUS = 5
SSIM_SIGMA = 1.5


def ssim(a, b):
    a, b = [(normalize(image.astype(np.float64)) + 1) / 2 for image in (a, b)]

    def local_mean(image):
        return gaussian_filter(image, sigma=SSIM_SIGMA, radius=SSIM_RADIUS, mode="reflect")

    ma, mb = local_mean(a), local_mean(b)
    va = local_mean(a * a) - ma**2
    vb = local_mean(b * b) - mb**2
    cov = local_mean(a * b) - ma * mb
    field = ((2 * ma * mb + 0.01**2) * (2 * cov + 0.03**2)
             / ((ma**2 + mb**2 + 0.01**2) * (va + vb + 0.03**2)))
    valid = slice(SSIM_RADIUS, -SSIM_RADIUS)
    return field[valid, valid].mean()


fashion_names = ["Tişört", "Pantolon", "Kazak", "Elbise", "Palto",
                 "Sandalet", "Gömlek", "Spor ayakkabı", "Çanta", "Bot"]
def class_name(domain, cls):
    return fashion_names[cls] if domain == "Fashion-MNIST" else str(cls)


score_rows, confidence_rows = [], []
for id_domain, ood_domain in [domains, domains[::-1]]:
    _, _, x_test, y_test = datasets[id_domain]
    ids = np.random.default_rng(73).choice(len(y_test), 190, replace=False)
    x = x_test[ids]
    targets, attributions = {}, {}
    for domain in domains:
        confidence, targets[domain] = predictions(models[domain], x).softmax(1).max(1)
        confidence_rows.append((id_domain, domain, confidence.mean().item()))
        maps = explain(models[domain], x, targets[domain], seed=56)
        np.testing.assert_array_equal(maps["SmoothGrad-SQ"], np.square(maps["SmoothGrad"]))
        attributions[domain] = maps
    sample = attributions[id_domain]["SmoothGrad"][0]
    np.testing.assert_allclose(ssim(sample, sample), 1.0, atol=1e-12)
    for method in methods:
        for idx, a, b in zip(ids, attributions[id_domain][method], attributions[ood_domain][method]):
            score_rows.append((id_domain, method, int(idx), ssim(a, b),
                               rank_similarity(a.ravel(), b.ravel())))
    fig, axes = plt.subplots(3, 1 + 2 * len(methods), figsize=(17, 4.8))
    for row, example in enumerate(range(3)):
        axes[row, 0].imshow(x[example, 0], cmap="gray")
        axes[row, 0].set_title(f"IDX {ids[example]}\n{class_name(id_domain, y_test[ids[example]].item())}",
                               fontsize=8)
        for j, domain in enumerate([id_domain, ood_domain]):
            group = "Alan içi" if j == 0 else "Alan dışı"
            cls = class_name(domain, targets[domain][example].item())
            for k, method in enumerate(methods):
                ax = axes[row, 1 + j * len(methods) + k]
                ax.imshow(normalize(attributions[domain][method][example]), cmap="RdBu_r", vmin=-1, vmax=1)
                if row == 0:
                    ax.set_title(method.replace(" ", "\n"), fontsize=8)
                if k == 0:
                    ax.set_xlabel(f"{group}\n{cls}", fontsize=8)
        for ax in axes[row]:
            ax.set(xticks=[], yticks=[])
    save_figure(fig, f"02-{id_domain.lower()}-atif.png")

scores = pd.DataFrame(score_rows, columns=["Girdi", "Yöntem", "IDX", "SSIM", "Spearman ρ"])
confidence_table = pd.DataFrame(confidence_rows,
    columns=["Girdi", "Modelin eğitim alanı", "Ortalama en yüksek olasılık"])
assert len(scores) == 2 * 190 * len(methods)
summary = scores.groupby(["Girdi", "Yöntem"]).agg(
    SSIM=("SSIM", "mean"), SSIM_SH=("SSIM", "sem"), Spearman=("Spearman ρ", "mean"),
    Spearman_SH=("Spearman ρ", "sem"), Geçerli=("Spearman ρ", "count"))
display(summary)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, metric in zip(axes, ["SSIM", "Spearman"]):
    for j, name in enumerate(domains):
        rows = summary.loc[name].reindex(methods)
        offsets = np.arange(len(methods)) + (j - 0.5) * 0.32
        ax.bar(offsets, rows[metric], 0.32, yerr=rows[metric + "_SH"], label=name, capsize=2)
    ax.set(xticks=np.arange(len(methods)), ylabel=metric, ylim=(-0.25, 1.05))
    ax.set_xticklabels(methods, rotation=35, ha="right")
    ax.axhline(0, color=".5", lw=0.8)
    ax.legend()
save_figure(fig, "03-benzerlik.png")

SSIM   SSIM_SH  Spearman  Spearman_SH  \
Girdi         Yöntem                                                       
Fashion-MNIST Girdi × gradyan  0.410853  0.010453  0.876447     0.006401   
              Gradyan          0.176911  0.006141  0.051065     0.007166   
              Guided Backprop  0.537520  0.005796  0.675868     0.006550   
              IG               0.189794  0.009854  0.039176     0.006756   
              SmoothGrad       0.106511  0.006585  0.052529     0.007454   
              SmoothGrad-SQ    0.325320  0.009156  0.056447     0.007764   
              VarGrad          0.386889  0.009078  0.205944     0.019093   
MNIST         Girdi × gradyan  0.609951  0.008078  0.988547     0.000525   
              Gradyan          0.112683  0.004830  0.071749     0.005675   
              Guided Backprop  0.572082  0.003694  0.553761     0.005065   
              IG               0.331851  0.012828  0.076031     0.010567   
              SmoothGrad       0.166998  0.006916  0.063346     0.006081   
              SmoothGrad-SQ    0.202746  0.007055  0.064435     0.005280   
              VarGrad          0.373070  0.006580  0.312673     0.010157   

                               Geçerli  
Girdi         Yöntem                    
Fashion-MNIST Girdi × gradyan      190  
              Gradyan              190  
              Guided Backprop      190  
              IG                   190  
              SmoothGrad           190  
              SmoothGrad-SQ        190  
              VarGrad              190  
MNIST         Girdi × gradyan      190  
              Gradyan              190  
              Guided Backprop      190  
              IG                   190  
              SmoothGrad           190  
              SmoothGrad-SQ        190  
              VarGrad              190

In [4]:
fmnist = summary.loc["Fashion-MNIST"]
squared = fmnist.loc["SmoothGrad-SQ"]
display(Markdown(f"""# Model açıklamaları için hata ayıklama testleri

**Bölüm 3 · Yorumlanabilirliğin amaçları · Özgün MNIST ve Fashion-MNIST**

## 1. Araştırma amacı

Adebayo, Muelly, Liccardi ve Kim (2020), dağılım dışı girdilerde atıf haritalarının model hatalarını ortaya çıkarıp çıkaramadığını inceler. Burada Bölüm 5, Tablo 3'ün Fashion-MNIST → MNIST karşılaştırması ve Ek I’de gösterilen ters yön uygulanır. [Makale ve ekleri](https://arxiv.org/html/2011.05429v1) · [Yazarların kodu](https://github.com/adebayoj/explaindebug)

## Seçilen Ödev Seçeneği: Seçenek 1 — Zafiyet/Manipülasyon

**Test zamanında dağılım kayması.** Aynı özgün Fashion-MNIST görüntüsü hem giysi hem MNIST rakam sınıflandırıcısına verilir. Piksel değerleri korunur ve her model kendi tahminini açıklar. Rakam modeli giysi görüntüsüne kendi sınıf uzayından bir rakam etiketi atar. Bu, modelin eğitim alanı dışındaki girdilere de tahmin üretmesinden kaynaklanan bir güvenilirlik zafiyetidir.

**Açıklamanın sınırı.** Fashion-MNIST girdilerinde iki modelin Guided Backprop haritaları arasındaki ortalama Spearman korelasyonu 0,676, Gradyan haritaları arasındaki korelasyon ise 0,051’dir. Guided Backprop haritası, modelin eğitim alanı değiştiğinde bile benzer piksel sıraları verebilir. Harita benzerliği, girdinin eğitim alanına uygunluğunu veya tahminin güvenilirliğini doğrulamak için yetersizdir.

Bu çalışma Seçenek 1’in zafiyet boyutuna, değiştirilmemiş test görüntüleriyle yapılan kontrollü dağılım kayması deneyi üzerinden karşılık gelir. [Adebayo ve ark., §5](https://arxiv.org/html/2011.05429v1#S5)

## 2. Veri ve uygulama

İki kaynağın özgün 60.000 eğitim ve 10.000 test görüntüsünü içeren dört gzip IDX arşivi ayrı ayrı indirilir ve doğrulanır. Görüntüler 28×28, tek kanallı ve [0, 1] ölçeğindedir. Tohum 41 ile her eğitim bölümünden 12.000 örnek seçilir. Her model kendi kaynağında 8 dönem Adam ile eğitilir ve alan içi doğruluk 10.000 test görüntüsünde ölçülür. Ortam {environment()}. [MNIST ve IDX kaynağı](https://storage.googleapis.com/cvdf-datasets/mnist/train-images-idx3-ubyte.gz) · [Fashion-MNIST veri kaynağı](https://github.com/zalandoresearch/fashion-mnist#get-the-data)

{training.to_markdown(index=False, floatfmt=".4f")}

![İki alanın eğitim eğrileri](sekiller/01-egitim.png)

## 3. Test zamanı müdahalesi ve ölçümler

Her yönde testten tohum 73 ile 190 görüntü seçilir. Bir görüntü iki modele de aynı piksel değerleriyle verilir. Her atıf, ilgili modelin kendi en yüksek logitini açıklar. Alan dışında tahmin edilmesi gereken ortak bir sınıf yoktur.

Yedi yöntem hesaplanır. Ek B uyarınca Gradyan, hedef logitin türevinin mutlak değeridir. Girdi × gradyan bu mutlak türevi piksel değeriyle çarpar. SmoothGrad, 50 gürültülü girdide hesaplanan işaretli gradyanın ortalamasıdır. SmoothGrad-SQ bu ortalamanın eleman bazında karesidir. VarGrad aynı 50 gradyanın piksel bazında popülasyon varyansıdır. Her görüntü için sıfır ortalamalı Gauss gürültüsünün standart sapması, Ek B'deki gibi, o görüntünün en büyük ve en küçük piksel değerleri farkının 0,15 katıdır. Aynı gürültü dizisi iki modelde tekrar kullanılır. IG, Ek B'deki en düşük değerli siyah referansı ve bu çalışmada seçilen 24 trapez aralığını kullanır. [Makale §2.2 ve Ek B](https://arxiv.org/html/2011.05429v1#A2)

SSIM öncesinde her harita kendi en yüksek mutlak değerine bölünür ve [0, 1] aralığına taşınır. Yerel ortalama, varyans ve kovaryans için standart sapması 1,5 ve yarıçapı tam 5 piksel olan 11×11 Gauss penceresi kullanılır. Son ortalamaya yalnızca pencerenin bütünüyle görüntü içinde kaldığı 18×18 konum girer. Önceki uygulamada SciPy'nin varsayılan yarıçapı 6 piksel olduğu için 13×13 filtre ile 5 piksellik kırpma kullanılıyordu. Spearman, her yöntemin ölçekleme öncesindeki atıf değerlerinin sırasını karşılaştırır. SH, görüntüler üzerinden ortalamanın standart hatasıdır.

## 4. Deneysel sonuçlar

{summary.reset_index().to_markdown(index=False, floatfmt=".3f")}

Fashion-MNIST girdilerinde Gradyan için SSIM {fmnist.loc["Gradyan", "SSIM"]:.3f} ve Spearman {fmnist.loc["Gradyan", "Spearman"]:.3f} ölçüldü. Guided Backprop için aynı değerler {fmnist.loc["Guided Backprop", "SSIM"]:.3f} ve {fmnist.loc["Guided Backprop", "Spearman"]:.3f} oldu. Düzeltilen SmoothGrad-SQ için SSIM {squared["SSIM"]:.3f} ve Spearman {squared["Spearman"]:.3f} oldu. Yapısal benzerlik ile piksel sıra uyumu yöntemlere göre ayrışabilir.

![Yöntem ve test alanına göre SSIM ve Spearman](sekiller/03-benzerlik.png)

![Aynı Fashion-MNIST girdilerinde alan içi ve dışı atıflar](sekiller/02-fashion-mnist-atif.png)

![Aynı MNIST girdilerinde alan içi ve dışı atıflar](sekiller/02-mnist-atif.png)

Her gösterim satırında iki modelin tahmini, ilgili atıf grubunun ilk panelinin altında bulunur. Mavi negatif ve kırmızı pozitif atıftır. Her harita kendi mutlak tepe değeri ile ölçeklenir.

{confidence_table.to_markdown(index=False, floatfmt=".3f")}

En yüksek softmax olasılığı betimleyici bir ölçümdür. Alan dışı doğruluk ya da kalibre edilmiş hata olasılığı olarak yorumlanmaz.

## 5. Makaleyle karşılaştırma ve sınırlılıklar

Yazarlar görsel benzerlik ile sıra benzerliğinin ayrışabildiğini raporlar. Tablo 3'te Fashion-MNIST → MNIST için Gradyan ve SmoothGrad sıra korelasyonları sıfıra yakındır, VarGrad ve Guided Backprop ise daha yüksek korelasyon gösterir. Yerel tabloda aynı yöntemlerin sıralaması ve iki ölçütün ayrışması görülebilir. Karşılaştırma yöntem örüntüleri düzeyindedir. [Bölüm 5, Tablo 3](https://arxiv.org/html/2011.05429v1#S5)

| Deney ayarı | Makale | Bu uygulama |
|---|---|---|
| Eğitim verisi | §5, MNIST ve Fashion-MNIST üzerine eğitilmiş BVD-CNN modellerini belirtir. Bu iki model için kullanılan eğitim örneği sayısı ve kimlikleri §5 ve Ek C'de verilmez. | Her özgün 60.000 görüntülük eğitim bölümünden tohum 41 ile seçilen 12.000 görüntü. Alan içi doğruluk özgün 10.000 örneklik test bölümünün tamamında ölçülür. |
| Mimari ve eğitim | Ek C, ikili BVD-CNN için beş evrişim, üç tam bağlı katman, ReLU ve son katmanda sigmoid belirtir. Adam, öğrenme hızı 0,001 ve 40 dönem ikili görev için verilir. | Beş 3×3 evrişim (16, 16, 32, 32, 48 kanal), iki 2×2 maksimum havuzlama ve 128, 64, 10 birimli üç tam bağlı katman. Adam 0,001, yığın 128, çapraz entropi ve 8 dönem. On sınıflı açıklamalar logit üzerinden hesaplanır. |
| Test kapsamı | Tablo 3’te 190 Fashion-MNIST görüntüsü ve üç alan dışı model vardır. Ek I’de ters yön için görsel örnek verilir. Ek C’de üç kanallı dönüşüm VGG-16 ile ilişkilendirilir. | Her yönde 190 test görüntüsü, tohum 73. İki tek kanallı CNN karşılaştırılır. Her model kendi en yüksek logitini açıklar. Yazarların örnek kimlikleri ve ağırlıkları elde edilemedi. |
| Atıf | Tablo 3’te 15 yöntem. Ek B, SmoothGrad varyantları ve VarGrad için 50 gürültülü örnek ve görüntü aralığının 0,15 katını belirtir. | Yedi yöntem ve aynı gürültü ayarları. PyTorch autograd, iki modelde ortak gürültü ve IG için 24 trapez aralığı kullanılır. |
| Benzerlik | Tablo 3’te SSIM, Spearman ve standart hata verilir. §5’te Gauss pencere parametreleri ile harita ölçekleme ayrıntıları belirtilmez. | Aynı ölçütler raporlanır. Tepeye göre ölçekleme ve 11×11 Gauss penceresi bu çalışmanın açıkça tanımlanmış seçimleridir. |

Tek eğitim tohumu kullanıldığı için standart hatalar görüntüler arasındaki değişkenliği yansıtır. Sonuçlar Bölüm 5’in kısmi deneysel yeniden üretimidir. [Makale §5 ve Ek B–C](https://arxiv.org/html/2011.05429v1#S5)

## 6. Kaynak denetimi

### Birds-vs-Dogs veri kökeni

| Bileşen | Doğrulanan bilgi |
|---|---|
| Kuş görüntüleri | Makalenin Ek C.1 bölümü Caltech-UCSD Birds-200-2011'i adlandırır. Resmî görüntüler ve ayrı piksel segmentasyonu arşivi mevcuttur. |
| Köpek görüntüleri | Ek C.1, Parkhi ve arkadaşlarının Cats and Dogs çalışmasını adlandırır. Oxford-IIIT Pet görüntüleri ve piksel düzeyinde trimap arşivi yayımlanır. Stanford Dogs, makalede ayrı kullanıcı çalışmasıyla ilişkilendirilir. |
| Arka plan | Bölüm 3.1 Places gökyüzünü kuşlarla, bambu ormanını köpeklerle eşler. Resmî deponun kullanıcı çalışması klasöründe beş gökyüzü ve beş bambu PNG'si bulunur. Bunların ikili deneydeki Places dosyalarıyla birebir eşlemesi ve her örnekte seçilen görsel doğrulanamadı. |
| Bölmeler | Ek C.1 sınıf başına 4.300 eğitim ve kalanın eşit doğrulama/test bölünmesini tarif eder. Resmî deponun yayımladığı altı dosya listesinde her sınıf için 4.300 eğitim, 500 doğrulama ve 190 test girdisi sayıldı. Sınıf başına 4.990 dosya adı benzersizdi. |
| Hazırlama ve maskeler | Kullanıcı çalışmasının ayrı kodu Oxford trimap 1 değerini ön plan olarak seçer. İkili deney için dosya–maske eşleştirmesi, arka plan seçim manifestosu ve birleştirme betiği yayımlanan dosyalardan doğrulanamadı. |

Bu nedenle Birds-vs-Dogs bu defterde uygulanmadı. Ayrıca Bölüm 5'teki ImageNet/VGG-16 kolu bu uygulamanın kapsamı dışında tutuldu.

- [Makale §3.1 ve Ek C–D](https://arxiv.org/html/2011.05429v1)
- [CUB-200-2011 görüntüleri](https://www.vision.caltech.edu/datasets/cub_200_2011/) · [CUB segmentasyon arşivi](https://data.caltech.edu/records/w9d68-gec53) · [Oxford-IIIT Pet görüntü ve trimap arşivleri](https://www.robots.ox.ac.uk/~vgg/data/pets/)
- [Places veri kaynağı](https://places2.csail.mit.edu/download.html) · [Yazarların ikili deney bölmeleri](https://github.com/adebayoj/explaindebug/tree/98f3b211c97c1bbbaec4d12cc8c102ecbbc1afe7/debuggingtests/data/datapartion) · [Ayrı kullanıcı çalışması kodu ve görselleri](https://github.com/adebayoj/explaindebug/tree/98f3b211c97c1bbbaec4d12cc8c102ecbbc1afe7/userstudy)
"""))

# Model açıklamaları için hata ayıklama testleri

**Bölüm 3 · Yorumlanabilirliğin amaçları · Özgün MNIST ve Fashion-MNIST**

## 1. Araştırma amacı

Adebayo, Muelly, Liccardi ve Kim (2020), dağılım dışı girdilerde atıf haritalarının model hatalarını ortaya çıkarıp çıkaramadığını inceler. Burada Bölüm 5, Tablo 3'ün Fashion-MNIST → MNIST karşılaştırması ve Ek I’de gösterilen ters yön uygulanır. [Makale ve ekleri](https://arxiv.org/html/2011.05429v1) · [Yazarların kodu](https://github.com/adebayoj/explaindebug)

## Seçilen Ödev Seçeneği: Seçenek 1 — Zafiyet/Manipülasyon

**Test zamanında dağılım kayması.** Aynı özgün Fashion-MNIST görüntüsü hem giysi hem MNIST rakam sınıflandırıcısına verilir. Piksel değerleri korunur ve her model kendi tahminini açıklar. Rakam modeli giysi görüntüsüne kendi sınıf uzayından bir rakam etiketi atar. Bu, modelin eğitim alanı dışındaki girdilere de tahmin üretmesinden kaynaklanan bir güvenilirlik zafiyetidir.

**Açıklamanın sınırı.** Fashion-MNIST girdilerinde iki modelin Guided Backprop haritaları arasındaki ortalama Spearman korelasyonu 0,676, Gradyan haritaları arasındaki korelasyon ise 0,051’dir. Guided Backprop haritası, modelin eğitim alanı değiştiğinde bile benzer piksel sıraları verebilir. Harita benzerliği, girdinin eğitim alanına uygunluğunu veya tahminin güvenilirliğini doğrulamak için yetersizdir.

Bu çalışma Seçenek 1’in zafiyet boyutuna, değiştirilmemiş test görüntüleriyle yapılan kontrollü dağılım kayması deneyi üzerinden karşılık gelir. [Adebayo ve ark., §5](https://arxiv.org/html/2011.05429v1#S5)

## 2. Veri ve uygulama

İki kaynağın özgün 60.000 eğitim ve 10.000 test görüntüsünü içeren dört gzip IDX arşivi ayrı ayrı indirilir ve doğrulanır. Görüntüler 28×28, tek kanallı ve [0, 1] ölçeğindedir. Tohum 41 ile her eğitim bölümünden 12.000 örnek seçilir. Her model kendi kaynağında 8 dönem Adam ile eğitilir ve alan içi doğruluk 10.000 test görüntüsünde ölçülür. Ortam Python 3.13.15, PyTorch 2.15.0+cpu, CPU. [MNIST ve IDX kaynağı](https://storage.googleapis.com/cvdf-datasets/mnist/train-images-idx3-ubyte.gz) · [Fashion-MNIST veri kaynağı](https://github.com/zalandoresearch/fashion-mnist#get-the-data)

| Modelin eğitim alanı   |   Özgün eğitim |   Kullanılan eğitim |   Özgün test |   Alan içi test doğruluğu |
|:-----------------------|---------------:|--------------------:|-------------:|--------------------------:|
| Fashion-MNIST          |          60000 |               12000 |        10000 |                    0.8539 |
| MNIST                  |          60000 |               12000 |        10000 |                    0.9760 |

![İki alanın eğitim eğrileri](sekiller/01-egitim.png)

## 3. Test zamanı müdahalesi ve ölçümler

Her yönde testten tohum 73 ile 190 görüntü seçilir. Bir görüntü iki modele de aynı piksel değerleriyle verilir. Her atıf, ilgili modelin kendi en yüksek logitini açıklar. Alan dışında tahmin edilmesi gereken ortak bir sınıf yoktur.

Yedi yöntem hesaplanır. Ek B uyarınca Gradyan, hedef logitin türevinin mutlak değeridir. Girdi × gradyan bu mutlak türevi piksel değeriyle çarpar. SmoothGrad, 50 gürültülü girdide hesaplanan işaretli gradyanın ortalamasıdır. SmoothGrad-SQ bu ortalamanın eleman bazında karesidir. VarGrad aynı 50 gradyanın piksel bazında popülasyon varyansıdır. Her görüntü için sıfır ortalamalı Gauss gürültüsünün standart sapması, Ek B'deki gibi, o görüntünün en büyük ve en küçük piksel değerleri farkının 0,15 katıdır. Aynı gürültü dizisi iki modelde tekrar kullanılır. IG, Ek B'deki en düşük değerli siyah referansı ve bu çalışmada seçilen 24 trapez aralığını kullanır. [Makale §2.2 ve Ek B](https://arxiv.org/html/2011.05429v1#A2)

SSIM öncesinde her harita kendi en yüksek mutlak değerine bölünür ve [0, 1] aralığına taşınır. Yerel ortalama, varyans ve kovaryans için standart sapması 1,5 ve yarıçapı tam 5 piksel olan 11×11 Gauss penceresi kullanılır. Son ortalamaya yalnızca pencerenin bütünüyle görüntü içinde kaldığı 18×18 konum girer. Önceki uygulamada SciPy'nin varsayılan yarıçapı 6 piksel olduğu için 13×13 filtre ile 5 piksellik kırpma kullanılıyordu. Spearman, her yöntemin ölçekleme öncesindeki atıf değerlerinin sırasını karşılaştırır. SH, görüntüler üzerinden ortalamanın standart hatasıdır.

## 4. Deneysel sonuçlar

| Girdi         | Yöntem          |   SSIM |   SSIM_SH |   Spearman |   Spearman_SH |   Geçerli |
|:--------------|:----------------|-------:|----------:|-----------:|--------------:|----------:|
| Fashion-MNIST | Girdi × gradyan |  0.411 |     0.010 |      0.876 |         0.006 |       190 |
| Fashion-MNIST | Gradyan         |  0.177 |     0.006 |      0.051 |         0.007 |       190 |
| Fashion-MNIST | Guided Backprop |  0.538 |     0.006 |      0.676 |         0.007 |       190 |
| Fashion-MNIST | IG              |  0.190 |     0.010 |      0.039 |         0.007 |       190 |
| Fashion-MNIST | SmoothGrad      |  0.107 |     0.007 |      0.053 |         0.007 |       190 |
| Fashion-MNIST | SmoothGrad-SQ   |  0.325 |     0.009 |      0.056 |         0.008 |       190 |
| Fashion-MNIST | VarGrad         |  0.387 |     0.009 |      0.206 |         0.019 |       190 |
| MNIST         | Girdi × gradyan |  0.610 |     0.008 |      0.989 |         0.001 |       190 |
| MNIST         | Gradyan         |  0.113 |     0.005 |      0.072 |         0.006 |       190 |
| MNIST         | Guided Backprop |  0.572 |     0.004 |      0.554 |         0.005 |       190 |
| MNIST         | IG              |  0.332 |     0.013 |      0.076 |         0.011 |       190 |
| MNIST         | SmoothGrad      |  0.167 |     0.007 |      0.063 |         0.006 |       190 |
| MNIST         | SmoothGrad-SQ   |  0.203 |     0.007 |      0.064 |         0.005 |       190 |
| MNIST         | VarGrad         |  0.373 |     0.007 |      0.313 |         0.010 |       190 |

Fashion-MNIST girdilerinde Gradyan için SSIM 0.177 ve Spearman 0.051 ölçüldü. Guided Backprop için aynı değerler 0.538 ve 0.676 oldu. Düzeltilen SmoothGrad-SQ için SSIM 0.325 ve Spearman 0.056 oldu. Yapısal benzerlik ile piksel sıra uyumu yöntemlere göre ayrışabilir.

![Yöntem ve test alanına göre SSIM ve Spearman](sekiller/03-benzerlik.png)

![Aynı Fashion-MNIST girdilerinde alan içi ve dışı atıflar](sekiller/02-fashion-mnist-atif.png)

![Aynı MNIST girdilerinde alan içi ve dışı atıflar](sekiller/02-mnist-atif.png)

Her gösterim satırında iki modelin tahmini, ilgili atıf grubunun ilk panelinin altında bulunur. Mavi negatif ve kırmızı pozitif atıftır. Her harita kendi mutlak tepe değeri ile ölçeklenir.

| Girdi         | Modelin eğitim alanı   |   Ortalama en yüksek olasılık |
|:--------------|:-----------------------|------------------------------:|
| Fashion-MNIST | Fashion-MNIST          |                         0.836 |
| Fashion-MNIST | MNIST                  |                         0.799 |
| MNIST         | Fashion-MNIST          |                         0.823 |
| MNIST         | MNIST                  |                         0.975 |

En yüksek softmax olasılığı betimleyici bir ölçümdür. Alan dışı doğruluk ya da kalibre edilmiş hata olasılığı olarak yorumlanmaz.

## 5. Makaleyle karşılaştırma ve sınırlılıklar

Yazarlar görsel benzerlik ile sıra benzerliğinin ayrışabildiğini raporlar. Tablo 3'te Fashion-MNIST → MNIST için Gradyan ve SmoothGrad sıra korelasyonları sıfıra yakındır, VarGrad ve Guided Backprop ise daha yüksek korelasyon gösterir. Yerel tabloda aynı yöntemlerin sıralaması ve iki ölçütün ayrışması görülebilir. Karşılaştırma yöntem örüntüleri düzeyindedir. [Bölüm 5, Tablo 3](https://arxiv.org/html/2011.05429v1#S5)

| Deney ayarı | Makale | Bu uygulama |
|---|---|---|
| Eğitim verisi | §5, MNIST ve Fashion-MNIST üzerine eğitilmiş BVD-CNN modellerini belirtir. Bu iki model için kullanılan eğitim örneği sayısı ve kimlikleri §5 ve Ek C'de verilmez. | Her özgün 60.000 görüntülük eğitim bölümünden tohum 41 ile seçilen 12.000 görüntü. Alan içi doğruluk özgün 10.000 örneklik test bölümünün tamamında ölçülür. |
| Mimari ve eğitim | Ek C, ikili BVD-CNN için beş evrişim, üç tam bağlı katman, ReLU ve son katmanda sigmoid belirtir. Adam, öğrenme hızı 0,001 ve 40 dönem ikili görev için verilir. | Beş 3×3 evrişim (16, 16, 32, 32, 48 kanal), iki 2×2 maksimum havuzlama ve 128, 64, 10 birimli üç tam bağlı katman. Adam 0,001, yığın 128, çapraz entropi ve 8 dönem. On sınıflı açıklamalar logit üzerinden hesaplanır. |
| Test kapsamı | Tablo 3’te 190 Fashion-MNIST görüntüsü ve üç alan dışı model vardır. Ek I’de ters yön için görsel örnek verilir. Ek C’de üç kanallı dönüşüm VGG-16 ile ilişkilendirilir. | Her yönde 190 test görüntüsü, tohum 73. İki tek kanallı CNN karşılaştırılır. Her model kendi en yüksek logitini açıklar. Yazarların örnek kimlikleri ve ağırlıkları elde edilemedi. |
| Atıf | Tablo 3’te 15 yöntem. Ek B, SmoothGrad varyantları ve VarGrad için 50 gürültülü örnek ve görüntü aralığının 0,15 katını belirtir. | Yedi yöntem ve aynı gürültü ayarları. PyTorch autograd, iki modelde ortak gürültü ve IG için 24 trapez aralığı kullanılır. |
| Benzerlik | Tablo 3’te SSIM, Spearman ve standart hata verilir. §5’te Gauss pencere parametreleri ile harita ölçekleme ayrıntıları belirtilmez. | Aynı ölçütler raporlanır. Tepeye göre ölçekleme ve 11×11 Gauss penceresi bu çalışmanın açıkça tanımlanmış seçimleridir. |

Tek eğitim tohumu kullanıldığı için standart hatalar görüntüler arasındaki değişkenliği yansıtır. Sonuçlar Bölüm 5’in kısmi deneysel yeniden üretimidir. [Makale §5 ve Ek B–C](https://arxiv.org/html/2011.05429v1#S5)

## 6. Kaynak denetimi

### Birds-vs-Dogs veri kökeni

| Bileşen | Doğrulanan bilgi |
|---|---|
| Kuş görüntüleri | Makalenin Ek C.1 bölümü Caltech-UCSD Birds-200-2011'i adlandırır. Resmî görüntüler ve ayrı piksel segmentasyonu arşivi mevcuttur. |
| Köpek görüntüleri | Ek C.1, Parkhi ve arkadaşlarının Cats and Dogs çalışmasını adlandırır. Oxford-IIIT Pet görüntüleri ve piksel düzeyinde trimap arşivi yayımlanır. Stanford Dogs, makalede ayrı kullanıcı çalışmasıyla ilişkilendirilir. |
| Arka plan | Bölüm 3.1 Places gökyüzünü kuşlarla, bambu ormanını köpeklerle eşler. Resmî deponun kullanıcı çalışması klasöründe beş gökyüzü ve beş bambu PNG'si bulunur. Bunların ikili deneydeki Places dosyalarıyla birebir eşlemesi ve her örnekte seçilen görsel doğrulanamadı. |
| Bölmeler | Ek C.1 sınıf başına 4.300 eğitim ve kalanın eşit doğrulama/test bölünmesini tarif eder. Resmî deponun yayımladığı altı dosya listesinde her sınıf için 4.300 eğitim, 500 doğrulama ve 190 test girdisi sayıldı. Sınıf başına 4.990 dosya adı benzersizdi. |
| Hazırlama ve maskeler | Kullanıcı çalışmasının ayrı kodu Oxford trimap 1 değerini ön plan olarak seçer. İkili deney için dosya–maske eşleştirmesi, arka plan seçim manifestosu ve birleştirme betiği yayımlanan dosyalardan doğrulanamadı. |

Bu nedenle Birds-vs-Dogs bu defterde uygulanmadı. Ayrıca Bölüm 5'teki ImageNet/VGG-16 kolu bu uygulamanın kapsamı dışında tutuldu.

- [Makale §3.1 ve Ek C–D](https://arxiv.org/html/2011.05429v1)
- [CUB-200-2011 görüntüleri](https://www.vision.caltech.edu/datasets/cub_200_2011/) · [CUB segmentasyon arşivi](https://data.caltech.edu/records/w9d68-gec53) · [Oxford-IIIT Pet görüntü ve trimap arşivleri](https://www.robots.ox.ac.uk/~vgg/data/pets/)
- [Places veri kaynağı](https://places2.csail.mit.edu/download.html) · [Yazarların ikili deney bölmeleri](https://github.com/adebayoj/explaindebug/tree/98f3b211c97c1bbbaec4d12cc8c102ecbbc1afe7/debuggingtests/data/datapartion) · [Ayrı kullanıcı çalışması kodu ve görselleri](https://github.com/adebayoj/explaindebug/tree/98f3b211c97c1bbbaec4d12cc8c102ecbbc1afe7/userstudy)
